# 🧲 Magnetar Colab

Torrent downloader → video converter → Google Drive uploader, with a Gradio web UI.

**How to use**
1. *Runtime → Change runtime type →* **T4 GPU** (recommended for conversions; CPU also works).
2. *Runtime → Run all*. Approve the Google Drive permission prompt.
3. Open the `gradio.live` link that appears at the bottom.

All the real code lives in the GitHub repo `kkdkavindu-beep/Torrent` — this notebook
mounts Drive, clones the latest version and launches it. Downloads go to `/content/downloads`,
processed videos land in **MyDrive/TorrentColab/** (configurable in the UI).
⏳ **Keep the last cell running** — it holds the Colab session open (it also injects a
keep-alive so free runtimes don't idle-disconnect while you use the Gradio link from
another tab). Interrupt the cell twice to shut down.

In [ ]:
# Cell 1 — mount Google Drive (the permission prompt appears here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — clone/pull the repo and install dependencies
import subprocess, sys, os

REPO = "https://github.com/kkdkavindu-beep/Torrent.git"
SRC = "/content/Torrent"

if os.path.isdir(os.path.join(SRC, ".git")):
    print("pulling latest code...")
    r = subprocess.run(["git", "-C", SRC, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        print("pull failed - re-cloning")
        subprocess.run(["rm", "-rf", SRC], check=True)
if not os.path.isdir(os.path.join(SRC, ".git")):
    print("cloning repo...")
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, SRC], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        raise RuntimeError("git clone failed - check the repo is public and exists")

os.chdir(SRC)
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
print("loaded commit:", head.stdout.strip())

print("installing dependencies (libtorrent, gradio)...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("pip install failed")

import libtorrent, gradio
print("libtorrent:", libtorrent.version)
print("gradio:", gradio.__version__)
print("setup complete ✓")

In [ ]:
# Cell 3 — launch the Gradio interface (keep this cell running; the app lives here)
import os, sys
SRC = SRC if "SRC" in globals() else "/content/Torrent"
os.chdir(SRC)
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Optional: protect the share link with a password
# os.environ["MAGNETAR_PASSWORD"] = "my-secret"

from magnetar_colab.app import launch
launch()